In [2]:
import pandas as pd
import re

# Load files
keywords = pd.read_csv("../CsvForDB/Keywords.csv")
drp = pd.read_csv("DRP.csv")

# Combine title + abstract into one searchable column
drp["search_text"] = (
    drp["title"].fillna("") + " " + drp["abstract"].fillna("")
).str.lower()

rows = []

for _, paper in drp.iterrows():
    text = paper["search_text"]

    for _, kw in keywords.iterrows():
        term = str(kw["normalized_term"]).strip().lower()

        if term:
            # Match only whole terms, allowing punctuation around them
            pattern = r"(?<![a-z0-9])" + re.escape(term) + r"(?![a-z0-9])"

            if re.search(pattern, text):
                rows.append({
                    "DRPId": paper["researchPortalId"],
                    "keywordID": kw["keywordId"]
                })

# Save result
result = pd.DataFrame(rows).drop_duplicates()
result.to_csv("../CsvForDB/DRPPaperKeywords.csv", index=False)

result.head()

,DRPId,keywordID
0,S01-000.00-438089,8
1,S01-000.00-438089,86
2,S01-000.00-438089,87
3,S01-000.00-438089,88
4,S01-000.00-438089,105


In [5]:
import pandas as pd

# Load file
df = pd.read_csv("DRP.csv")

# Select columns and rename researchPortalId → DRPId
df_new = df[[
    "researchPortalId",
    "year",
    "title",
    "type",
    "abstract",
    "file",
    "totalNumberOfContributors"
]].rename(columns={
    "researchPortalId": "DRPId"
})

# Save to new CSV
df_new.to_csv("../CsvForDB/DRPPaper.csv", index=False)

# Preview result
df_new.head()

,DRPId,year,title,type,abstract,file,totalNumberOfContributors
0,S01-000.00-438089,2022,"""Affective Publics"" : Performing Trust on Dani...",Journal Article,"In Denmark, as with elsewhere in the world, Tw...",10.1086/719645,5
1,S01-000.00-829674,2025,"""Danske reaktioner på grønlandsk selvstyre, se...",Book,NaN,https://pure.diis.dk/ws/files/27733344/Dilemma...,2
2,S01-000.00-482896,2016,"""Det lille vand"" : russernes forhold til vodka...",Book Chapter,NaN,http://e-pages.dk/ku/1205/2,3
3,S01-000.00-098927,2021,"""Does Vinegar Kill Coronavirus?"" - Using Searc...",Conference Paper,Health experts and government authorities' act...,https://vbn.aau.dk/ws/files/393455744/iConf202...,2
4,S01-000.00-444355,2013,"""European Spallation Source - Technical Design...",Report Chapter,NaN,NaN,20


In [7]:
import csv
import pandas as pd

people = set()

with open("DRP.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    
    for row in reader:
        # Get both columns safely
        author_1 = row.get("author", "")
        author_2 = row.get("authors", "")
        
        # Combine them
        combined = []
        if author_1:
            combined.extend(author_1.split(";"))
        if author_2:
            combined.extend(author_2.split(";"))
        
        # Clean and store unique names
        for person in combined:
            person = person.strip()
            if person:
                people.add(person)

# Split "Last, First" into separate columns
rows = []
for i, person in enumerate(sorted(people), start=1):
    parts = [p.strip() for p in person.split(",", 1)]
    
    if len(parts) == 2:
        last_name, first_name = parts[0], parts[1]
    else:
        last_name, first_name = person, ""
    
    rows.append([i, first_name, last_name])

# Create dataframe
df_people = pd.DataFrame(rows, columns=["DRPPersonId", "firstName", "lastName"])

# Save CSV
df_people.to_csv("../CsvForDB/DRPPerson.csv", index=False)

# Show first rows
df_people.head(20)

,DRPPersonId,firstName,lastName
0,1,PRIS study group,&
1,2,Leen M.,'T Hart
2,3,Peter A C,'T Hoen
3,4,Leen M,'t Hart
4,5,Leen M.,'t Hart
5,6,Nils A.,'t Hart
6,7,Peter A C,'t Hoen
7,8,Andrea,'t Mannetje
8,9,Det Fragmentariske Institut for Komparative Ti...,(FIKT)
9,10,International Multiple Sclerosis Genetics Cons...,(IMSGC)


In [9]:
import pandas as pd

# 1. Load data
drp = pd.read_csv("../CsvForDB/DRPPerson.csv")
person = pd.read_csv("../CsvForDB/person.csv")

# 2. Define normalization
def normalize(s):
    return str(s).lower().replace(".", "").strip()

# 3. Create normalized columns (👉 THIS is where your code goes)
drp["firstName_norm"] = drp["firstName"].apply(normalize)
drp["lastName_norm"] = drp["lastName"].apply(normalize)

person["firstName_norm"] = person["firstName"].apply(normalize)
person["lastName_norm"] = person["lastName"].apply(normalize)

# 4. Merge using normalized columns (instead of original ones)
merged = drp.merge(
    person[["UUID", "firstName_norm", "lastName_norm"]],
    on=["firstName_norm", "lastName_norm"],
    how="left"
)

# 5. Count changes
changed_count = (merged["UUID"].notna() & (merged["DRPPersonId"] != merged["UUID"])).sum()

# 6. Replace IDs
merged["DRPPersonId"] = merged["UUID"].combine_first(merged["DRPPersonId"])

# 7. Clean up helper columns
merged = merged.drop(columns=["UUID", "firstName_norm", "lastName_norm"])

# 8. Save
merged.to_csv("../CsvForDB/DRPPerson.csv", index=False)

print(f"Number of IDs changed: {changed_count}")

Number of IDs changed: 335


In [10]:
import csv
import pandas as pd

# Load DRPPerson.csv
df_people = pd.read_csv("../CsvForDB/DRPPerson.csv")

# Make a lookup: "Last, First" -> person info
person_lookup = {}
for _, row in df_people.iterrows():
    full_name = f"{str(row['lastName']).strip()}, {str(row['firstName']).strip()}".strip()
    person_lookup[full_name] = {
        "DRPPersonId": row["DRPPersonId"],
        "personName": row["firstName"],
        "personLastName": row["lastName"]
    }

# Build DRPPaperContributors rows
rows = []

with open("DRP.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        drp_id = row.get("researchPortalId", "")
        author = row.get("author", "")
        authors = row.get("authors", "")

        contributors = []

        if author:
            contributors.extend(author.split(";"))
        if authors:
            contributors.extend(authors.split(";"))

        # Clean names and remove duplicates within same paper
        contributors = list(dict.fromkeys([c.strip() for c in contributors if c.strip()]))

        for contributor in contributors:
            if contributor in person_lookup:
                rows.append({
                    "DRPId": drp_id,
                    "DRPPersonId": person_lookup[contributor]["DRPPersonId"],
                    "personName": person_lookup[contributor]["personName"],
                    "personLastName": person_lookup[contributor]["personLastName"]
                })
            else:
                # Optional: keep unmatched names visible
                parts = [p.strip() for p in contributor.split(",", 1)]
                last_name = parts[0] if len(parts) > 0 else ""
                first_name = parts[1] if len(parts) > 1 else ""

                rows.append({
                    "DRPId": drp_id,
                    "DRPPersonId": "",
                    "personName": first_name,
                    "personLastName": last_name
                })

# Create dataframe
df_contributors = pd.DataFrame(rows)

# Save CSV
df_contributors.to_csv("../CsvForDB/DRPPaperContributors.csv", index=False)

# Preview
df_contributors.head(50)

,DRPId,DRPPersonId,personName,personLastName
0,S01-000.00-438089,cac3823d-3dc6-4513-8448-be21a2d49344,Tobias,Pedersen
1,S01-000.00-438089,16424,Samantha Dawn,Breslin
2,S01-000.00-438089,13581,Anders,Blok
3,S01-000.00-438089,36650,Thyge Ryom,Enggaard
4,S01-000.00-438089,50725,Tobias,Gårdhus
5,S01-000.00-438089,110333,Morten Axel,Pedersen
6,S01-000.00-829674,07a52bc0-1368-4c96-b20d-4bf9668a3693,Henrik,Larsen
7,S01-000.00-829674,63005,Uffe,Jakobsen
8,S01-000.00-482896,82e44c10-e721-4308-8c18-cbe0f5ece50f,Jesper,Nielsen
9,S01-000.00-482896,42031,Kim,Frederichsen
